# YOLOv8 Arabic Line Detector — Training Notebook

Originally run on Kaggle. Trains YOLOv8n on synthetic 2500×2500 pages.

**Steps:**
1. Generate synthetic pages (or use `scripts/make_synthetic_pages.py`).
2. Run the training cell.
3. Evaluate on the two test images.

Training details: epochs=100, patience=10, imgsz=1280, fliplr=0.0, batch=16
Final val mAP50=0.995, mAP50-95=0.972, precision=0.998, recall=0.999

**Data paths:** Adjust paths below to match your environment.


In [ ]:
!pip install ultralytics

In [ ]:
import pandas as pd
from ultralytics import YOLO

### Data loading

Download and unzip your generated_pics.zip here, or use:
```bash
python scripts/make_synthetic_pages.py ...
```
Then set the path below to your output directory.

### Data loading

Download and unzip your generated_pics.zip here, or use:
```bash
python scripts/make_synthetic_pages.py ...
```
Then set the path below to your output directory.

In [ ]:
import zipfile
import os

zip_path = "./output/generated_pics.zip"
extract_path = "./output/generated_pics"

if os.path.exists(zip_path) and zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    print("Extraction successful!")
    print(os.listdir(extract_path)[:10])
else:
    print("The downloaded file is not a valid ZIP archive.")

In [ ]:
!cat -A "./output/generated_pics/arabic_lines.yaml"
yaml_lines = [
    'path: "./output/generated_pics"',
    "train: images/train",
    "val: images/val",
    "",
    "names:",
    "  0: printed_line",
    "  1: handwritten_line",
]

with open("./output/generated_pics/arabic_lines.yaml", "w") as f:
    f.write("\n".join(yaml_lines) + "\n")

In [ ]:
model = YOLO("yolov8n.pt")
model.train(
    data="./output/generated_pics/arabic_lines.yaml",
    epochs = 100 ,
    imgsz = 1280 ,
    patience=10,
    fliplr =0.0 ,
    project="./output/runs",
    name="line_detector"
)

In [ ]:
from IPython.display import FileLink, display

best_model = "./output/runs/line_detector-3/weights/last.pt"

display(FileLink(best_model))

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pt", ".png", ".jpg", ".jpeg")):
            print(os.path.join(root, f))

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("models/best.pt")   # use your real path

test_images = [
    "/kaggle/input/datasets/mohamedhamada37/line-detector/6c4e76d745630bdedeed0ffd2975b3c7.jpg",   # use your real paths
    "/kaggle/input/datasets/mohamedhamada37/line-detector/8459f20c2251a6fe00e323d38bc22330.jpg",
]

for path in test_images:
    res = model.predict(path, imgsz=1280, conf=0.4, iou=0.5)
    plt.figure(figsize=(10, 14))
    plt.imshow(res[0].plot()[..., ::-1])
    plt.axis("off")
    plt.title(f"{os.path.basename(path)}: {len(res[0].boxes)} boxes")
    plt.show()

In [ ]:
CODE2CHAR = {
    # letters
    'aa':'ا','ba':'ب','ta':'ت','th':'ث','ja':'ج','ha':'ح','kh':'خ',
    'da':'د','dh':'ذ','ra':'ر','za':'ز','se':'س','sh':'ش','sa':'ص',
    'de':'ض','to':'ط','zha':'ظ','ay':'ع','gh':'غ','fa':'ف','ka':'ق',
    'ke':'ك','la':'ل','ma':'م','na':'ن','he':'ه','wa':'و','ya':'ي',
    # hamza / special forms
    'ae':'أ','ah':'إ','am':'آ','al':'ئ','wl':'ؤ','hh':'ء','ee':'ى',
    'tee':'ة','teE':'ة',
    # lam-alef ligatures
    'laaa':'لا','laae':'لأ','laah':'لإ','laam':'لآ',
    # space and punctuation
    'sp':' ','dot':'.','com':'،','col':':','scr':'؛','dbq':'"',
    'bro':'(','brc':')','hyp':'-','per':'%','exc':'!',
    'fsl':'/','bsl':'\\','equ':'=','qts':'؟',
}
# digits: n0..n9 -> '0'..'9'


def code_cahr(rea):
    
